# Reinforcement Learning w Human Feedback and Transfer Learning to LLama

Install random stuff idk bru also run this command to insall llama3.1 select one 
```
huggingface-cli download meta-llama/Meta-Llama-3.1-405B --local-dir Meta-Llama-3.1-405B
huggingface-cli download meta-llama/Meta-Llama-3.1-70B --local-dir Meta-Llama-3.1-70B
huggingface-cli download meta-llama/Meta-Llama-3.1-8B --local-dir Meta-Llama-3.1-8B

```

In [1]:
from transformers import LlamaForCausalLM, AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
import gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecEnv

C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Load pre-trained LLaMA model and tokenizer
model_path = "E:\\models\\Meta-Llama-3.1-8B"
model = LlamaForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Set a pad_token if not already set (using eos_token as pad_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # You can also add a custom token like '[PAD]'

# Load detection model (RoBERTa) for detecting GPT-generated text
detection_model = AutoModelForSequenceClassification.from_pretrained("openai-community/roberta-base-openai-detector")

C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\accelerate\utils\modeling.py:1462: UserWarning: Current model requires 4224 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 4/4 [00:16<00:00,  4.07s/it]
Some weights of the model checkpoint at openai-community/roberta-base-openai-detector were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exa

In [10]:
# Reward function
def detect_gpt_generated_text(text):
    # Tokenize the input and add padding + attention_mask
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']  # Attention mask to ignore padding tokens

    # Perform inference with the detection model
    with torch.no_grad():
        outputs = detection_model(input_ids=input_ids, attention_mask=attention_mask)
    
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    
    # Index 0: Not GPT-generated, Index 1: GPT-generated
    prob_gpt_generated = 1 - probabilities[0][1].item()
    return prob_gpt_generated

def reward_function(text, detection_model):
    detectability_score = detect_gpt_generated_text(text)
    reward = -detectability_score
    return reward

In [11]:
# Define RL environment
class TextGenerationEnv(gym.Env):
    def __init__(self, model, detection_model):
        super(TextGenerationEnv, self).__init__()
        self.action_space = gym.spaces.Discrete(tokenizer.vocab_size)
        self.observation_space = gym.spaces.Box(low=-float('inf'), high=float('inf'), shape=(1,), dtype=np.float32)
        self.model = model
        self.detection_model = detection_model

    def step(self, action):
        # Generate text based on the tokenized action
        input_ids = torch.tensor([[action]]).to(model.device)  # Action as token ID
        generated_output = model.generate(input_ids, max_length=50, do_sample=True)
        generated_text = tokenizer.decode(generated_output[0], skip_special_tokens=True)

        # Calculate reward
        reward = reward_function(generated_text, self.detection_model)
        done = True  # Episode ends after one step for simplicity
        return generated_text, reward, done, {}

    def reset(self):
        # Reset the environment (for simplicity, just return 0 as a placeholder)
        return np.array([0], dtype=np.float32)

In [12]:
# Wrap the environment
env = DummyVecEnv([lambda: TextGenerationEnv(model, detection_model)])

# Train using PPO
ppo = PPO('MlpPolicy', env, verbose=1)
ppo.learn(total_timesteps=10000)

# Define optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

# Number of epochs for fine-tuning
num_epochs = 5

# Use the trained policy to generate and fine-tune the model
for _ in range(num_epochs):
    generated_text = ...  # Generate text using the model (you need to replace this ellipsis with logic)
    inputs = tokenizer(generated_text, return_tensors="pt", padding=True, truncation=True)
    
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']  # Pass the attention mask along
    
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    loss = -reward_function(generated_text, detection_model)
    
    loss.backward()
    optimizer.step()

Using cuda device


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


IndexError: index out of range in self